<h1>Chapter 10 - Coding Agents</h1>
<i>Create an Agent for developing code.</i>


<a href="..."><img src="https://img.shields.io/badge/Buy%20the%20Book!-grey?logo=amazon"></a>
<a href="..."><img src="https://img.shields.io/badge/O'Reilly-white.svg?logo=data:image/svg%2bxml;base64,PHN2ZyB3aWR0aD0iMzQiIGhlaWdodD0iMjciIHZpZXdCb3g9IjAgMCAzNCAyNyIgZmlsbD0ibm9uZSIgeG1sbnM9Imh0dHA6Ly93d3cudzMub3JnLzIwMDAvc3ZnIj4KPGNpcmNsZSBjeD0iMTMiIGN5PSIxNCIgcj0iMTEiIHN0cm9rZT0iI0Q0MDEwMSIgc3Ryb2tlLXdpZHRoPSI0Ii8+CjxjaXJjbGUgY3g9IjMwLjUiIGN5PSIzLjUiIHI9IjMuNSIgZmlsbD0iI0Q0MDEwMSIvPgo8L3N2Zz4K"></a>
<a href="..."><img src="https://img.shields.io/badge/GitHub%20Repository-black?logo=github"></a>
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](...)

---

This notebook is for Chapter 10 of the [An Illustrated Guide to AI Agents](...) book by [Maarten Grootendorst](https://www.linkedin.com/in/mgrootendorst/) and [Jay Alammar](https://www.linkedin.com/in/jalammar).

---

<a href="...">
<img src="https://learning.oreilly.com/covers/urn:orm:book:9798341662681/400w/" width="350"/></a>


### **[OPTIONAL]** - Installing Packages on Google Colab <img src="https://upload.wikimedia.org/wikipedia/commons/d/d0/Google_Colaboratory_SVG_Logo.svg" width=100>

If you are viewing this notebook on Google Colab (or any other cloud vendor), you need to **uncomment and run** one of the following codeblock to install the dependencies for this chapter. If you want to use a cloud provider, you only need to run the following code block:

In [1]:
# %%capture
# !pip install illustrated-agents

---

💡 **NOTE**: If you want to use the GPU with `ollama`, then you will have to select a GPU first. In Google Colab, go to
**Runtime > Change runtime type > Hardware accelerator > GPU > GPU type > T4**. 

Then, **uncomment** and run this codeblock:

---

In [2]:
# !apt-get install -y zstd > /dev/null 2>&1 && curl -fsSL https://ollama.com/install.sh | sh
# !nohup ollama serve > /dev/null 2>&1 & sleep 3 && ollama pull gemma4:e4b &

<hr style="height: 5px; border: none; border-radius: 5px; background: linear-gradient(to right, #000000, #7D7D7D);" />

## 1 - Choosing Your LLM

At the beginning of every chapter, we start by choosing the LLM that we want to use:

In [1]:
from illustrated_agents.chapters.ch2 import LLM

# Gemma 4 E4B (with native thinking and tool calling)
llm = LLM(model="gemma4:e4b", think=True)

If you want to use another LLM, here are a couple of options that use the `OpenAI` library:

In [2]:
# from openai import OpenAI
# from illustrated_agents.llm import OpenAIClientLLM

# # Ollama through OpenAI API
# client = OpenAI(base_url="http://localhost:11434/v1/", api_key="no_key")
# llm = OpenAIClientLLM(model="gemma4:e4b", client=client, think=True)

# # Llama.cpp server
# client = OpenAI(base_url="http://localhost:8080/v1/", api_key="no_key")
# llm = OpenAIClientLLM(model="gemma-4-E4B-it-Q4_K_M", client=client, think=True)

# # LM Studio
# client = OpenAI(base_url="http://localhost:1234/v1/", api_key="no_key")
# llm = OpenAIClientLLM(model="gemma-4-e4b-it", client=client, think=True)

# # Google's Gemini / Gemma
# client = OpenAI(base_url="https://generativelanguage.googleapis.com/v1beta/openai/", api_key="YOUR_GEMINI_API_KEY")
# llm = OpenAIClientLLM(model="gemini-2.5-flash", client=client, think=True)

## 2 - Coding Tools


As we covered in this chapter, code tools are what tie an LLM to a software system. The two main categories are :

* **file manipulation tools**: This lets the agent read, list, and write files
* **code interpreter**: This lets the agent execute code it writes. 

Below, we define four tools that cover both categories: `read_file` and `list_files` allow the agent to explore existing files, `write_file` lets it create or modify files, and `execute_python` runs Python code in a subprocess with a timeout. Finally, we also add the `show_python` function that allows the agent to show formatted code.



In [3]:
import subprocess
import sys
from pathlib import Path

# # For the `show_python` function
# from pygments import highlight
# from pygments.lexers import PythonLexer
# from pygments.formatters import TerminalFormatter

def read_file(path: str) -> str:
    """Read a file's contents."""
    target = Path(path)
    if not target.exists():
        return f"Error: '{path}' not found."
    return target.read_text(encoding="utf-8")


def list_files(directory: str = ".") -> str:
    """List files in a directory."""
    target = Path(directory)
    if not target.is_dir():
        return f"Error: '{directory}' is not a directory."
    entries = sorted(target.iterdir())
    return "\n".join(p.name + ("/" if p.is_dir() else "") for p in entries) or "(empty)"


def write_file(path: str, content: str) -> str:
    """Write content to a file."""
    target = Path(path)
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(content, encoding="utf-8")
    return f"Written to '{path}'."


def execute_python(code: str) -> str:
    """Execute Python code and return output."""
    try:
        result = subprocess.run(
            [sys.executable, "-c", code],
            capture_output=True,
            text=True,
            timeout=30,
        )
        if result.returncode != 0:
            return f"Exit code {result.returncode}\nSTDOUT:\n{result.stdout}\nSTDERR:\n{result.stderr}".strip()
        return result.stdout.strip() or "(no output)"
    except subprocess.TimeoutExpired:
        return "Error: Code execution timed out (30s limit)."

# def show_python(code: str):
#     """Print syntax-highlighted Python code."""
#     print(highlight(code, PythonLexer(), TerminalFormatter()))
#     return "Shown code with syntax highlighting."

Note that the `return` part of every function is essentially the `OBSERVATION` that is being returned to your `TinyAgent`!

Next, we define these tools and add them to your `NativeTools`:

In [4]:
from illustrated_agents.chapters.ch5 import NativeTools

tools = NativeTools()
tools.add_tool("read_file", read_file)
tools.add_tool("list_files", list_files)
tools.add_tool("write_file", write_file)
tools.add_tool("execute_python", execute_python)
# tools.add_tool("show_python", show_python)

## 3 - Too Much Autonomy?

Although we now have a number of tools, some of them can really mess with your enviroment. Writing to files and especially executing python functions is something that shouldn't be done arbritarily. As such, we are going to update `Tools` and `NativeTools` with a simple safeguard. The safeguard will be that certain tools will require approval before they can be run. LLMs are not perfect and might accidentally do things they really shouldn't do, such as updating their own configurations thereby breaking it... true story!

Either way, this update is rather straightforward, the updated `Tools` class is now as follows but note that we only made marginal changes (more and that later!):

The only change we made is to `run` to check whether a tool requires approval. If it does, then the user will need to reply with either `"y"` or `"yes"`:

Let's explore it a bit closer:

In [5]:
from illustrated_agents.chapters.ch5 import execute_tool_annotated; execute_tool_annotated

Now we can redefine the `NativeTools` by having it inherit this behavior again from `Tools` which now uses `requires_approval` to check which functions require specific approval from the user before they can be run.

Let's redefine our `NativeTools` and this time make sure that the `write` and `execute` tools require approval:

In [6]:
tools = NativeTools(requires_approval=["write_file", "execute_python"])
tools.add_tool("read_file", read_file)
tools.add_tool("list_files", list_files)
tools.add_tool("write_file", write_file)
tools.add_tool("execute_python", execute_python)
# tools.add_tool("show_python", show_python)

Note that the write and execute tools are marked as `requires_approval`. As discussed before, this means that everytime your `TinyAgent` wants to run that particular, it will ask confirmation from you before continuing. 

Let's try it out and see if it would try to execute some python code:

In [7]:
from illustrated_agents.chapters.ch2 import Response

# Define tool calll
response = Response(
    content="Some content...",
    reasoning="Some reasoning...",
    tool_call= {
        "tool": "execute_python",
        "kwargs": {"code": "print('Hello World!')"}
    }
)

# Execute Tool
tools.execute(response)

Allow execute_python? [y/N]  


"Tool 'execute_python' was denied by the user."

## 4 - The Styling

To create a coding agent, you will need a nicely looking interface for us to use so you can easily see the models thoughts, actions, and observations. For that, we make use of the [`rich`](https://github.com/Textualize/rich) package library for rich text and beautiful formatting in the terminal.

We create a `Display` class that animates variious parts of the thinking/answering process. This `Display` is merely used for styling:

In [230]:
BLACK = "\033[30m"
BOLD = "\033[1m"
DIM = "\033[2m"
RESET = "\033[0m"
YELLOW = "\033[33m"
GREEN = "\033[32m"
RED = "\033[31m"
PURPLE = "\033[35m"

class Display:
    """Handles agent events with ANSI-styled terminal output."""

    def __call__(self, event: str, data: str | Response = None) -> None:

        # "Thinking" line (static, with a random message)
        if event == "thinking":
            print(f"  {DIM}Thinking...{RESET}")

        # Print THOUGHT
        elif event == "response":
            print(
                f"  {BOLD}{GREEN}{'THOUGHT':<13}{RESET}"
                f"{BLACK}{data.reasoning}{RESET}"
            )
            if data.content:
                print(
                    f"  {BOLD}{PURPLE}{'ANSWER':<13}{RESET}"
                    f"{BLACK}{data.content}{RESET}"
                )

        # Print ACTION
        elif event == "tool_call" and data:
            tool = data.tool_call["tool"]
            kwargs = data.tool_call["kwargs"]
            print(
                f"  {BOLD}{RED}{'ACTION':<13}{RESET}{BLACK}{tool}({kwargs}){RESET}"
            )

        # Print OBSERVATION
        elif event == "observation":
            print(
                f"  {BOLD}{YELLOW}{'OBSERVATION':<13}{RESET}{BLACK}{data}{RESET}\n"
            )
            print(f"{DIM}{'─' * 80}{RESET}\n")

In [262]:
class Display:
    """Handles agent events with ANSI-styled terminal output."""

    def __call__(self, event: str, data: str | Response = None) -> None:

        # "Thinking" line
        if event == "thinking":
            print(f"  {DIM}Thinking...\n{RESET}")

        # Print THOUGHT
        elif event == "response":
            print(f"{BOLD}{GREEN}{'▒▒ THOUGHT ▒▒':<13}{RESET}")
            print(f"{BLACK}{data.reasoning}{RESET}\n")
            if data.content:
                print(f"{BOLD}{PURPLE}{'▒▒ ANSWER ▒▒':<13}{RESET}")
                print(f"{BLACK}{data.content}{RESET}\n")

        # Print ACTION
        elif event == "tool_call" and data:
            tool = data.tool_call["tool"]
            kwargs = data.tool_call["kwargs"]
            print(f"{BOLD}{RED}{'▒▒ ACTION ▒▒':<13}{RESET}")
            print(f"{BLACK}{tool}({kwargs}){RESET}\n")

        # Print OBSERVATION
        elif event == "observation":
            print(f"{BOLD}{YELLOW}{'▒▒ OBSERVATION ▒▒':<13}{RESET}")
            print(f"{BLACK}{data}{RESET}\n")
            print(f"{DIM}{'─' * 80}{RESET}\n")

Let's take a closer look at what it is doing and why:

In [263]:
# from illustrated_agents.chapters.ch11 import display_annotated; display_annotated

To further illustrate this `Display`, we can mimic the behavior of the Agent and have it "generate" a `THOUGHT`, `ACTION`, `TOOL_CALL`, and `OBSERVATION`.

In [264]:
# Let's create a Response with a tool call and no final answer
response = Response(
    content="",
    reasoning="Let's execute some python!",
    tool_call= {
        "tool": "execute_python",
        "kwargs": {"code": "print('Hello World!')"}
    }
)


# Display the Response
display = Display()
display("thinking", response)
display("response", response)
display("tool_call", response)
display("observation", "Hello World!")

  Thinking...

▒▒ THOUGHT ▓▓
Let's execute some python!

▒▒ ACTION ▒▒ 
execute_python({'code': "print('Hello World!')"})

▒▒ OBSERVATION ▒▒
Hello World!

────────────────────────────────────────────────────────────────────────────────



We can also show what the final answer would look like:

In [257]:
# Let's create a Response with a final answer and no tool call
response = Response(
    content="I believe that one plus one is two!",
    reasoning="I should answer the question.",
)

# Display the Response
display = Display()
display("thinking", response)
display("response", response)

  Thinking...

▒▒ THOUGHT ▒▒
I should answer the question.

▒▒ ANSWER ▒▒ 
I believe that one plus one is two!



## 5 - The `TinyAgent`

The `TinyAgent` will also need an update so that it can use the `Display` whenever it is finished creating `THOUGHT`, `ACTION`, `TOOL_CALL`, and `OBSERVATION`.

The `TinyAgent` only requires adding `self.display(...)` at various places to track what is happening

Since the changes to the `TinyAgent` are minimal and requires adding the `Display` and calling it in specific places whenever the `TinyAgent` has completed something.

In [258]:
from illustrated_agents.chapters.ch10 import tinyagents_diff; tinyagents_diff

Which makes the `TinyAgent` the following updated class:

In [259]:
from illustrated_agents.chapters.ch2 import Response, Trajectory
from illustrated_agents.chapters.ch4 import Memory
from illustrated_agents.chapters.ch5 import Tools
from illustrated_agents.chapters.ch6 import ReAct

class TinyAgent:
    """A minimal, modular, and educational agent framework."""

    def __init__(
        self,
        llm: LLM,
        memory: Memory,
        tools: Tools,
        planner: ReAct,
        display: Display,
    ):
        self.llm = llm
        self.memory = memory
        self.tools = tools
        self.planner = planner
        self.display = display

        self.trajectory = Trajectory()

        # Build system prompt with all components
        system_prompt = "You are a helpful assistant.\n\n"
        system_prompt += self.planner.prompt
        system_prompt += self.tools.prompt
        self.memory.add("system", system_prompt)

    def run(self, task: str, image_data: str = None) -> str:
        """Run the agent on a task."""
        self.memory.add("user", task, image_data=image_data)
        self.trajectory.initialize(task)

        # *Autonomy* loop
        for step in range(self.planner.max_steps):
            result = self._step()
            if result is not None:
                return result

        return "Max steps reached without completion."

    def _step(self) -> str | None:
        """Perform a single step."""
        # THOUGHT: Generate response and add to memory
        response = self.llm.generate(
            self.memory.get_messages(), tools=self.tools.schemas
        )
        self.memory.add(
            "assistant", response.content, tool_call=response.tool_call
        )
        self.display("response", response)

        # Tool parsing
        response = self.planner.parse(response)
        response = self.tools.parse(response)

        # ANSWER: Stopping mechanism
        if self.tools.is_done(response):
            self.trajectory.add(response)
            return response.content

        return self._execute_action(response)

    def _execute_action(self, response: Response) -> None:
        """Execute a tool action."""

        # ACTION: execute tools
        self.display("tool_call", response)
        result = self.tools.execute(response)

        # OBSERVATION: add tool results to memory and display
        role, observation = self.tools.observation(result)
        self.memory.add(role, observation)
        self.trajectory.add(response, observation)
        self.display("observation", observation)

        return None

## 6 - Putting It All Together

In [260]:
from illustrated_agents.chapters.ch6 import NativeReAct
from illustrated_agents.chapters.ch9 import MultimodalMemory

# Multimodal Agent
agent = TinyAgent(
    llm=llm, 
    tools=tools, 
    memory=MultimodalMemory(),
    planner=NativeReAct(),
    display=display
)

In [261]:
# Run your Coding Agent
## NOTE: You will be asked to allow the agent to execute Python code. 
## Check the output and type `y` if you want to allow it.
# _ = agent.run("Create a simple function that uses `Rich` to show formatted Python. Execute it before writing it to rich_example.py")

_ = agent.run("List all files in this folder and create a simple function that calls LLM().")

▒▒ THOUGHT ▒▒
The user has two requests:
1. List all files in the current folder.
2. Create a simple function that calls `LLM()`.

For the first request, I should use the `list_files` tool. Since no specific directory is provided, I will assume the current directory.

For the second request, creating a function that calls `LLM()` is a programmatic task. I should use the `execute_python` tool to write and execute this function definition.

I will address these requests sequentially.

**Step 1: List files in the current directory.**
I will call `list_files({})` (assuming the current directory is the default/intended directory for an empty input). Since the tool signature for `list_files` takes an optional `directory` string, I will call it without arguments if allowed, or with an empty string/'.'. Looking at the definition, it seems safer to assume the default directory is handled if I provide a specific path, but if I want "this folder", an empty call or calling it with `.` is appropria

Allow write_file? [y/N]  y


▒▒ OBSERVATION ▒▒
Written to 'new_llm_caller.py'.

────────────────────────────────────────────────────────────────────────────────

▒▒ THOUGHT ▒▒
None

▒▒ ANSWER ▒▒ 
The files in the current directory are:
*   `.ipynb_checkpoints/`
*   `__pycache__/`
*   `calculate_derivative.py`
*   `chapter10.ipynb`
*   `llm_caller.py`
*   `llm_function.py`
*   `llm_utils.py`
*   `llm_wrapper.py`
*   `new_llm_call.py`
*   `README.md`
*   `simple_llm_call.py`
*   `simple_llm_interface.py`

I have also created a new file named **`new_llm_caller.py`** containing a simple function `call_llm()` that simulates calling an LLM. You can review its contents or execute the file further if needed.<channel|>The files in the current directory are:
*   `.ipynb_checkpoints/`
*   `__pycache__/`
*   `calculate_derivative.py`
*   `chapter10.ipynb`
*   `llm_caller.py`
*   `llm_function.py`
*   `llm_utils.py`
*   `llm_wrapper.py`
*   `new_llm_call.py`
*   `README.md`
*   `simple_llm_call.py`
*   `simple_llm_interface.py

And there you have it! Your own Coding Agent that can use various tools, run python code, and even allows you to judge whether you want the code to be executed. As always, let's explore the messages:

In [165]:
from illustrated_agents.utils import TrajectoryViewer
TrajectoryViewer(agent.trajectory)

In [166]:
def main():
    print("TinyAgent ready. Type 'exit' to quit.\n")

    while True:
        try:
            query = input("> ").strip()
        except (KeyboardInterrupt, EOFError):
            break
        if not query or query.lower() in ("exit", "quit"):
            break
        try:
            result = agent.run(query)
            print(f"\nANSWER: {result}\n")
        except Exception as e:
            print(f"ERROR: {e}\n")


if __name__ == "__main__":
    main()


TinyAgent ready. Type 'exit' to quit.



>  exit


# What We Built

In this chapter, we covered a major step into creating a helpful assistant, namely by adding coding capabilities! 

In [20]:
from illustrated_agents.chapters.ch11 import what_we_built; what_we_built

╭───────────────────────────────────────────────── What We Built ─────────────────────────────────────────────────╮
│ TinyAgent                                                                                                       │
│ ├── agent.py      ← Updated (Added printing to the `Display` to see its intermediate steps.)                    │
│ ├── display.py    ← New (A new `Display` class to format the agent's THOUGHTS, ACTIONS, and OBSERVATIONS.)      │
│ ├── llm.py                                                                                                      │
│ ├── memory.py                                                                                                   │
│ ├── planning.py                                                                                                 │
│ ├── skills.py                                                                                                   │
│ ├── toolbox.py    ← Updated (Added tools for Coding Agents.)                                                    │
│ ├── tools.py      ← Updated (Added XML-based tool parsing and calling with human-in-the-loop checks.)           │
│ └── trajectory.py                                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯